## VAE Demo on the EMNIST Dataset

In [29]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils.dataset import load_data
from utils.layers import VAE
from utils.losses import vae_loss
from utils.optim import Adam
from utils.save import save_model
from utils.tensor import Tensor


In [30]:
X_train, y_train, X_test, y_test = load_data()

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: torch.Size([112800, 784])
Test set shape: torch.Size([18800, 784])


In [31]:
config = {"input_size": 784, "hidden_size": 64, "latent_size": 16}
model = VAE(**config)
optimizer = Adam(model.parameters())

In [32]:
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    batch_size = 64
    perm = np.random.permutation(len(X_train))
    pbar = tqdm(range(0, len(X_train), batch_size), desc=f"Epoch {epoch + 1}/{epochs}")
    for start in pbar:
        idx = perm[start : start + batch_size]
        x = Tensor(X_train[idx])
        optimizer.zero_grad()
        x_hat, mean, logvar = model(x)
        loss = vae_loss(x, x_hat, mean, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.data
        seen = start + len(idx)
        pbar.set_postfix(loss=f"{total_loss / seen:.2f}")
save_model(model, config, "vae.json")
print("Saved model to vae.json")

Epoch 10/10: 100%|██████████| 1763/1763 [00:03<00:00, 518.53it/s, loss=139.86]


Saved model to vae.json
